In [1]:
!pip install groq gradio pypdf python-docx --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 8.3 MB/s eta 0:00:00


In [3]:
import gradio as gr
from groq import Groq
from pypdf import PdfReader
import docx

# Paste your Groq API key here
GROQ_API_KEY = "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"

client = Groq(api_key=GROQ_API_KEY)


def extract_text_from_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"

    return text


def extract_text_from_docx(file_path):
    document = docx.Document(file_path)
    text = ""

    for paragraph in document.paragraphs:
        text += paragraph.text + "\n"

    return text


def analyze_resume(resume_file, job_role):
    try:
        if GROQ_API_KEY == "GROQ KEY":
            return "Please paste your actual Groq API key."

        if resume_file is None:
            return "Please upload a resume."

        if job_role is None or job_role.strip() == "":
            return "Please enter a target job role."

        file_path = resume_file.name

        if file_path.lower().endswith(".pdf"):
            resume_text = extract_text_from_pdf(file_path)

        elif file_path.lower().endswith(".docx"):
            resume_text = extract_text_from_docx(file_path)

        else:
            return "Please upload only PDF or DOCX file."

        if resume_text.strip() == "":
            return "Could not extract text from the resume. Try another PDF/DOCX file."

        prompt = f"""
You are an expert HR recruiter and career advisor.

Analyze the following resume for this target role:

Target Role:
{job_role}

Resume:
{resume_text}

Give the output in this exact format:

1. Overall Resume Summary
2. Key Strengths
3. Skill Gaps
4. Missing Keywords
5. Job Fit Score out of 100
6. Resume Improvement Suggestions
7. Suggested Professional Summary
8. Interview Preparation Topics

Use simple, clear, and professional language.
"""

        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.3,
            max_tokens=900
        )

        return response.choices[0].message.content

    except Exception as e:
        return "Error occurred: " + str(e)


demo = gr.Interface(
    fn=analyze_resume,
    inputs=[
        gr.File(
            label="Upload Resume",
            file_types=[".pdf", ".docx"]
        ),
        gr.Textbox(
            label="Target Job Role",
            placeholder="Example: Assistant Professor in ECE, Data Scientist, AI Engineer"
        )
    ],
    outputs=gr.Textbox(
        lines=22,
        label="Resume Analysis Report"
    ),
    title="AI Resume Analyzer",
    description="Upload your resume and enter a target job role. The app analyzes strengths, gaps, keywords, and job-fit score."
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f8bda3fd1c7b3a5683.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://f8bda3fd1c7b3a5683.gradio.live
